In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from anthropic import Anthropic

client = Anthropic()

model = "claude-haiku-4-5-20251001"

In [20]:
def add_messages(messages, text):
    message = {"role": "user", "content": text}
    messages.append(message)

def add_assistant_messages(messages, text):
    message = {"role": "assistant", "content": text}
    messages.append(message)

def chat(messages, system = "", temperature = 0, stop_sequences = []):
    message = client.messages.create(
    model=model,
    max_tokens=500,
    messages=messages,
    system=system,
    temperature=temperature,
    stop_sequences=stop_sequences
    )
    return message.content[0].text.strip()

In [55]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "python or regex or json",
        "solution_criteria": "Key criterias to evaluate the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages=[]
    add_messages(messages, prompt)
    add_assistant_messages(messages, "```json")
    return chat(messages=messages, stop_sequences=["```"])

In [56]:
dataset = generate_dataset()
dataset

'[\n    {\n        "task": "Parse an AWS S3 bucket name from a full S3 URI (e.g., \'s3://my-bucket-name/path/to/object\') and extract only the bucket name",\n        "format": "regex",\n        "solution_criteria": "Should correctly extract bucket names from various S3 URI formats, handle URIs with and without paths, and work with bucket names containing hyphens and numbers"\n    },\n    {\n        "task": "Write a Python function that takes an AWS CloudWatch log event and extracts the timestamp and message fields, returning them as a dictionary",\n        "format": "python",\n        "solution_criteria": "Function should handle the standard CloudWatch Logs JSON structure, gracefully handle missing fields, and return a clean dictionary with timestamp and message keys"\n    },\n    {\n        "task": "Create a JSON configuration object for an AWS Lambda function that specifies runtime as Python 3.11, timeout of 60 seconds, memory of 512 MB, and includes environment variables for API_KEY

In [57]:
with open("test_dataset.json", "w") as f:
    json.dump(json.loads(dataset), f, indent = 2)

In [51]:
def run_prompt(test_case):
    """Merges the prompt and the test case input and returns the result"""
    prompt = f"""Please solve the following task: {test_case["task"]} 
    * Response should only contain Python, JSON or plain Regex
    * Do not add any comments, commentary and explaination
    """
    messages = []
    add_messages(messages, prompt)
    add_assistant_messages(messages, "```code")
    return chat(messages, stop_sequences = ["```"])

In [43]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python_syntax(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_by_syntax(text, text_format):
    match text_format:
        case 'python':
            return validate_python_syntax(text)
        case 'json':
            return validate_json(text)
        case 'regex':
            return validate_regex(text)
        case '_':
            return 0        

In [58]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>


Criteria you should use to Evaluate the Solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages=[]
    add_messages(messages, eval_prompt)
    add_assistant_messages(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [59]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_by_syntax(output, test_case["format"])

    score = (model_score + syntax_score) / 2
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [60]:
from statistics import mean

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results

In [61]:
with open("test_dataset.json", "r") as f:
    dataset = json.load(f)

run_eval(dataset)

Average Score: 8.333333333333334


[{'output': "import re\n\ndef extract_s3_bucket_name(s3_uri):\n    match = re.match(r's3://([a-zA-Z0-9._-]+)(?:/|$)', s3_uri)\n    if match:\n        return match.group(1)\n    return None",
  'test_case': {'task': "Parse an AWS S3 bucket name from a full S3 URI (e.g., 's3://my-bucket-name/path/to/object') and extract only the bucket name",
   'format': 'regex',
   'solution_criteria': 'Should correctly extract bucket names from various S3 URI formats, handle URIs with and without paths, and work with bucket names containing hyphens and numbers'},
  'score': 8.5,
  'reasoning': 'The solution successfully addresses the core requirement of extracting bucket names from various S3 URI formats and handles the specified cases (hyphens, numbers, with/without paths). However, it lacks strict adherence to AWS S3 bucket naming conventions. The regex pattern is permissive and would accept technically invalid bucket names. For production use, additional validation constraints should be added to en